# Clasificación de pastores alemanes y otros perros

Este notebook prepara el dataset de imágenes para el problema binario de detección de pastores alemanes.

La idea es que todo quede parametrizado para que mañana puedas cambiar la raza objetivo sin rehacer el flujo completo.

## Decisión sobre el código preliminar

El bloque que venía en este notebook estaba escrito dentro de una celda markdown, así que no era reutilizable tal cual.

En lugar de copiarlo sin más, conviene transformarlo en un flujo parametrizado con estas etapas:

1. Descarga de al menos 4000 imágenes por clase.
2. Normalización a 224x224.
3. Eliminación de duplicados con `imagededup`.
4. Organización por carpetas para facilitar el cambio de clase en el futuro.

La configuración de clases, palabras clave y rutas queda centralizada en una sola sección.

In [ ]:
from pathlib import Path
from typing import Dict, Tuple

PROJECT_ROOT = Path('.').resolve()
DATA_ROOT = PROJECT_ROOT / 'data' / 'dog_classification'
RAW_ROOT = DATA_ROOT / 'raw'
PROCESSED_ROOT = DATA_ROOT / 'processed'
TARGET_SIZE: Tuple[int, int] = (224, 224)
MIN_IMAGES_PER_CLASS = 4000

CLASS_SPECS: Dict[str, Dict[str, str]] = {
    'german_shepherd': {
        'keyword': 'German Shepherd dog',
        'raw_dir': str(RAW_ROOT / 'german_shepherd'),
        'processed_dir': str(PROCESSED_ROOT / 'german_shepherd'),
    },
    'other_dogs': {
        'keyword': 'dog breeds',
        'raw_dir': str(RAW_ROOT / 'other_dogs'),
        'processed_dir': str(PROCESSED_ROOT / 'other_dogs'),
    },
}

for spec in CLASS_SPECS.values():
    Path(spec['raw_dir']).mkdir(parents=True, exist_ok=True)
    Path(spec['processed_dir']).mkdir(parents=True, exist_ok=True)

for folder in (DATA_ROOT, RAW_ROOT, PROCESSED_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

print('Ruta base del proyecto:', DATA_ROOT)
print('Tamaño objetivo:', TARGET_SIZE)
print('Clases configuradas:', ', '.join(CLASS_SPECS))

In [ ]:
# Si faltan dependencias, instálalas antes de ejecutar estas celdas:
# %pip install icrawler pillow imagededup tqdm

from pathlib import Path
from typing import Iterable

from icrawler.builtin import BingImageCrawler, GoogleImageCrawler
from PIL import Image, ImageOps
from imagededup.methods import PHash
from tqdm.auto import tqdm

SEARCH_ENGINES = {
    'google': GoogleImageCrawler,
    'bing': BingImageCrawler,
}


def crawl_images(keyword: str, output_dir: Path, max_num: int, engine: str = 'google') -> None:
    """Descarga imágenes para una clase concreta."""
    crawler_cls = SEARCH_ENGINES[engine]
    crawler = crawler_cls(storage={'root_dir': str(output_dir)})
    crawler.crawl(keyword=keyword, max_num=max_num)


def resize_image(image_path: Path, target_size: tuple[int, int]) -> None:
    """Convierte a RGB y ajusta el tamaño sin deformar la imagen."""
    with Image.open(image_path) as image:
        cleaned = ImageOps.fit(image.convert('RGB'), target_size, method=Image.Resampling.LANCZOS)
        cleaned.save(image_path, quality=95)


def resize_folder(folder: Path, target_size: tuple[int, int]) -> None:
    for image_path in tqdm(list(folder.rglob('*'))):
        if image_path.is_file() and image_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
            resize_image(image_path, target_size)


def remove_duplicates(folder: Path) -> None:
    """Elimina duplicados dentro de una carpeta usando imagededup."""
    deduper = PHash()
    duplicate_map = deduper.find_duplicates(image_dir=str(folder), scores=False)
    duplicate_names = {duplicate_name for duplicates in duplicate_map.values() for duplicate_name in duplicates}

    for duplicate_name in duplicate_names:
        candidate = folder / duplicate_name
        if candidate.exists():
            candidate.unlink()


def build_dataset(class_specs: dict[str, dict[str, str]], minimum_images: int, target_size: tuple[int, int]) -> None:
    for class_name, spec in class_specs.items():
        raw_dir = Path(spec['raw_dir'])
        processed_dir = Path(spec['processed_dir'])
        keyword = spec['keyword']

        raw_dir.mkdir(parents=True, exist_ok=True)
        processed_dir.mkdir(parents=True, exist_ok=True)

        print(f'[{class_name}] descargando {minimum_images} imágenes con el texto: {keyword}')
        crawl_images(keyword=keyword, output_dir=raw_dir, max_num=minimum_images)

        print(f'[{class_name}] ajustando tamaño a {target_size}')
        resize_folder(raw_dir, target_size)

        print(f'[{class_name}] eliminando duplicados')
        remove_duplicates(raw_dir)

        print(f'[{class_name}] listo en {raw_dir}')


# Ejecución de referencia.
# build_dataset(CLASS_SPECS, MIN_IMAGES_PER_CLASS, TARGET_SIZE)